# Cross-Asset & Factor Correlation Structure — Beta-Adjusted Residuals

**The problem with naive factor correlations.** Data providers (Koyfin among them)
build factor proxies as raw ETF spreads — Value is `IWD − SPY`, Momentum is
`MTUM − SPY`, and so on. That subtraction silently assumes the factor ETF has a
**beta of exactly 1** to the market. It almost never does. Whatever market
exposure isn't cancelled by the `− SPY` term leaks straight into the "factor"
stream, and because *every* proxy carries the same leftover market beta, they all
co-move. Build a correlation matrix out of those streams and you are mostly
measuring the market correlating with itself.

**What we do instead.** For each ETF we run a *rolling* regression against SPY and
keep the **residual** — the part of the return that the market move at that moment
does **not** explain (`α + ε`). This is the beta-adjusted excess return. A residual
of zero means "this ETF did exactly what its recent beta said it should, given
where SPY went." A non-zero residual is genuine idiosyncratic / factor movement.

Correlations between these residual streams therefore answer a sharper question:
*once you strip out the common market pulse, what still moves together?* That is
where real factor, sector, and regional structure lives — and it's where the
"correlations have collapsed" / dispersion thesis can actually be tested.

This notebook is self-contained: it installs its dependencies, downloads and caches
the data, and runs end-to-end with no editing.

## Setup

Installs run only if the import fails, so re-executing the notebook on a configured
machine is a no-op.

In [ ]:
# Install dependencies only if missing (safe to re-run).
import importlib, subprocess, sys

def _ensure(pkg, import_name=None):
    try:
        importlib.import_module(import_name or pkg)
    except ImportError:
        print(f"installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for _pkg, _imp in [("yfinance", "yfinance"), ("pandas", "pandas"),
                   ("numpy", "numpy"), ("matplotlib", "matplotlib"),
                   ("seaborn", "seaborn"), ("pyarrow", "pyarrow")]:
    _ensure(_pkg, _imp)
print("dependencies ready")

## CONFIG

Everything tunable lives here: the rolling window, the sample dates, the cache
behaviour, and the full ETF universe. The universe is a plain dict so it is trivial
to add or drop a ticker — each entry carries a human-readable label and an asset-class
group, and the **order of groups here drives the sort order of every heatmap** below.

**Why a 63-day window?** ~63 trading days ≈ 3 calendar months. Long enough that the
rolling beta is stable rather than noise, short enough that it tracks regime shifts
instead of averaging across them. It is exposed as a parameter so you can stress it.

In [ ]:
import os
from datetime import date

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

# ----- core parameters -------------------------------------------------------
WINDOW      = 63              # rolling regression window (~3 months)
START_DATE  = "2020-01-01"
END_DATE    = None           # None -> today
BENCHMARK   = "SPY"          # regression RHS for residualization

# ----- caching ---------------------------------------------------------------
CACHE_PATH    = "prices_cache.parquet"
FORCE_REFRESH = False        # set True to bypass the cache and re-download

# ----- universe: ticker -> (label, group) ------------------------------------
# Group order here defines the categorical sort used in every heatmap.
UNIVERSE = {
    # STYLE FACTORS
    "MTUM": ("Momentum",        "Style Factors"),
    "IJR":  ("Small-Cap",       "Style Factors"),
    "IWD":  ("Value",           "Style Factors"),
    "VYM":  ("High Div Yield",  "Style Factors"),
    "IWF":  ("Growth",          "Style Factors"),
    "USMV": ("Low Vol",         "Style Factors"),
    "QUAL": ("Quality",         "Style Factors"),
    "GURU": ("Hedge Funds",     "Style Factors"),
    "PKW":  ("Buybacks",        "Style Factors"),
    "IVE":  ("Large Value",     "Style Factors"),
    "IWN":  ("Small Value",     "Style Factors"),
    # US SECTORS (SPDR)
    "XLF":  ("Financials",      "US Sectors"),
    "XLK":  ("Technology",      "US Sectors"),
    "XLE":  ("Energy",          "US Sectors"),
    "XLV":  ("Health Care",     "US Sectors"),
    "XLI":  ("Industrials",     "US Sectors"),
    "XLY":  ("Cons. Discr.",    "US Sectors"),
    "XLP":  ("Cons. Staples",   "US Sectors"),
    "XLU":  ("Utilities",       "US Sectors"),
    "XLB":  ("Materials",       "US Sectors"),
    "XLRE": ("Real Estate",     "US Sectors"),
    "XLC":  ("Comm. Services",  "US Sectors"),
    # INTERNATIONAL
    "EFA":  ("EAFE",            "International"),
    "VWO":  ("Emerging Mkts",   "International"),
    "EEM":  ("EM (alt)",        "International"),
    "EWJ":  ("Japan",           "International"),
    "EWG":  ("Germany",         "International"),
    "FEZ":  ("Eurozone",        "International"),
    "EWC":  ("Canada",          "International"),
    "EWW":  ("Mexico",          "International"),
    "EWZ":  ("Brazil",          "International"),
    "EWY":  ("South Korea",     "International"),
    "INDA": ("India",           "International"),
    "FXI":  ("China LC",        "International"),
    "KWEB": ("China Internet",  "International"),
    "ASHR": ("China A-shares",  "International"),
    # COMMODITIES
    "GLD":  ("Gold",            "Commodities"),
    "SLV":  ("Silver",          "Commodities"),
    "USO":  ("Oil",             "Commodities"),
    "UNG":  ("Nat Gas",         "Commodities"),
    "DBA":  ("Agriculture",     "Commodities"),
    "DBB":  ("Base Metals",     "Commodities"),
    "CPER": ("Copper",          "Commodities"),
}

GROUP_ORDER = ["Style Factors", "US Sectors", "International", "Commodities"]
LABELS  = {t: lbl for t, (lbl, _) in UNIVERSE.items()}
GROUPS  = {t: grp for t, (_, grp) in UNIVERSE.items()}
ALL_TICKERS = list(UNIVERSE.keys()) + [BENCHMARK]

end_resolved = END_DATE or date.today().isoformat()
print(f"{len(UNIVERSE)} ETFs + benchmark {BENCHMARK} | "
      f"{START_DATE} -> {end_resolved} | window={WINDOW}d")

### Plot theme

A consistent dark theme keeps the deck legible when shared. Cyan `#00FFF7` and amber
`#FFA629` are the accent pair used throughout. Space Mono is used for tick labels and
Inter for body text *if installed* — otherwise matplotlib's defaults are used, so the
notebook never errors on a machine without those fonts.

In [ ]:
ACCENT_CYAN  = "#00FFF7"
ACCENT_AMBER = "#FFA629"

_available = {f.name for f in fm.fontManager.ttflist}
_body = next((f for f in ("Inter", "Helvetica Neue", "DejaVu Sans") if f in _available), None)
_mono = next((f for f in ("Space Mono", "DejaVu Sans Mono") if f in _available), None)
if "Space Mono" not in _available or "Inter" not in _available:
    print("note: Space Mono / Inter not installed - falling back to defaults gracefully")

plt.style.use("dark_background")
mpl.rcParams.update({
    "figure.facecolor":  "#0d1117",
    "axes.facecolor":    "#0d1117",
    "savefig.facecolor": "#0d1117",
    "axes.edgecolor":    "#30363d",
    "grid.color":        "#21262d",
    "grid.alpha":        0.6,
    "axes.grid":         True,
    "axes.titlesize":    13,
    "axes.titleweight":  "bold",
    "axes.labelcolor":   "#c9d1d9",
    "text.color":        "#c9d1d9",
    "xtick.color":       "#8b949e",
    "ytick.color":       "#8b949e",
    "figure.dpi":        110,
})
if _body:
    mpl.rcParams["font.family"] = _body
DIVERGING = sns.diverging_palette(28, 187, s=90, l=55, as_cmap=True)  # amber<->cyan

def _style_ticks(ax):
    if _mono:
        for lbl in (*ax.get_xticklabels(), *ax.get_yticklabels()):
            lbl.set_fontfamily(_mono)

print(f"theme set | body={_body} | mono={_mono}")

## Data: download & cache

yfinance is the source. Downloads are cached to a local parquet file, so the slow
network round-trip happens once; subsequent runs read from disk in milliseconds. Flip
`FORCE_REFRESH = True` in CONFIG to re-pull (e.g. to extend the sample to today).

In [ ]:
import yfinance as yf

def load_prices(tickers, start, end, cache_path=CACHE_PATH, force=FORCE_REFRESH):
    '''Adjusted close panel for `tickers`, cached to parquet.'''
    if (not force) and os.path.exists(cache_path):
        cached = pd.read_parquet(cache_path)
        if set(tickers).issubset(cached.columns):
            sub = cached[tickers].loc[start:(end or cached.index.max())]
            if not sub.dropna(how="all").empty:
                print(f"loaded {len(sub.columns)} tickers from cache ({cache_path})")
                return sub
    print("downloading from yfinance ...")
    raw = yf.download(tickers, start=start, end=end, auto_adjust=True,
                      progress=False)["Close"]
    if isinstance(raw, pd.Series):
        raw = raw.to_frame()
    raw = raw.dropna(how="all")
    raw.to_parquet(cache_path)
    print(f"downloaded & cached {len(raw.columns)} tickers -> {cache_path}")
    return raw

prices = load_prices(ALL_TICKERS, START_DATE, END_DATE)
prices = prices[[c for c in ALL_TICKERS if c in prices.columns]]
print(prices.shape, "|", prices.index.min().date(), "->", prices.index.max().date())
prices.tail(3)

## Log returns

We use **log returns** throughout. They are additive across time, behave better in
the OLS that follows, and keep the residual algebra clean. The regression is run on
log returns and the residuals inherit those units.

In [ ]:
log_ret = np.log(prices).diff().dropna(how="all")

# Drop tickers with insufficient history (need a healthy multiple of WINDOW).
min_obs = WINDOW * 3
counts = log_ret.count()
insufficient = counts[counts < min_obs].index.tolist()
for t in insufficient:
    print(f"WARNING: dropping {t} ({LABELS.get(t, t)}) - "
          f"{counts[t]} obs < {min_obs} required")
keep = [c for c in log_ret.columns if c not in insufficient]
log_ret = log_ret[keep]

assert BENCHMARK in log_ret.columns, f"benchmark {BENCHMARK} missing from data"
print(f"\n{len(log_ret.columns)} tickers retained, {len(log_ret)} daily obs")

## Residualization — the core step

For every non-SPY ticker we estimate a univariate OLS of its return on SPY's return
over a trailing window, then take the residual at the window's endpoint.

For a one-variable regression the OLS coefficients have a closed form, so instead of
fitting ~50 separate `RollingOLS` models we apply the formulas as **vectorized rolling
operations** — same math, far faster:

$$\beta_t = \frac{\mathrm{Cov}_W(r_i, r_{SPY})}{\mathrm{Var}_W(r_{SPY})},\qquad
\alpha_t = \bar r_{i,W} - \beta_t\,\bar r_{SPY,W}$$

$$\varepsilon_t = r_{i,t} - (\alpha_t + \beta_t\, r_{SPY,t})$$

Two things matter and are easy to get wrong:

1. **The window ends at `t` — it is not centered.** `β_t` and `α_t` are estimated
   from data up to and including `t`, and the residual is then evaluated at `t` using
   the *actual* returns at `t`. Using `center=True` would leak future information.
2. **This is a fitted residual at the endpoint, not a smoothed series.** We are not
   averaging anything — we are asking "given the beta SPY-relationship just exhibited,
   how far off was today's move?" That deviation is the factor signal.

In [ ]:
def residualize(returns, benchmark=BENCHMARK, window=WINDOW):
    '''Rolling beta-adjusted residuals (alpha + epsilon) of each column vs benchmark.

    Vectorized closed-form univariate OLS; window ENDS at t (not centered).'''
    rb = returns[benchmark]
    cols = [c for c in returns.columns if c != benchmark]

    var_b  = rb.rolling(window).var()
    mean_b = rb.rolling(window).mean()

    out = {}
    for c in cols:
        ri = returns[c]
        beta  = ri.rolling(window).cov(rb) / var_b
        alpha = ri.rolling(window).mean() - beta * mean_b
        out[c] = ri - (alpha + beta * rb)
    res = pd.DataFrame(out, index=returns.index)
    # First `window-1` rows are NaN (incomplete window); drop them.
    return res.dropna(how="all")

residuals = residualize(log_ret)
print("residuals:", residuals.shape,
      "|", residuals.index.min().date(), "->", residuals.index.max().date())
residuals.tail(3).iloc[:, :6]

---
## 1 · Static heatmap — full-sample residual correlations

This is the structural snapshot: across the entire sample, which residual streams
move together once the market is stripped out? Read it in blocks. Strong within-group
color (e.g. the sector block) is expected; the *interesting* signal is cross-block —
where a style factor lines up with a region, or commodities decouple from everything.
Tickers are sorted by the group order from CONFIG, with white separators at the group
boundaries. The colormap is diverging and pinned to [−1, 1] so zero is always neutral.

In [ ]:
def _ordered(tickers):
    '''Sort tickers by CONFIG group order, then by appearance within the group.'''
    present = [t for t in residuals.columns if t in tickers]
    return sorted(present, key=lambda t: (GROUP_ORDER.index(GROUPS[t]),
                                          list(UNIVERSE).index(t)))

def _group_boundaries(ordered):
    '''Index positions where the group changes (for separator lines).'''
    bounds, seen = [], None
    for i, t in enumerate(ordered):
        if GROUPS[t] != seen and i != 0:
            bounds.append(i)
        seen = GROUPS[t]
    return bounds

def plot_corr_heatmap(corr, title, ax=None, annot=False):
    ordered = _ordered(corr.columns)
    corr = corr.loc[ordered, ordered]
    disp  = [LABELS[t] for t in ordered]
    if ax is None:
        size = max(8, len(ordered) * 0.34)
        _, ax = plt.subplots(figsize=(size, size))
    sns.heatmap(corr, cmap=DIVERGING, center=0, vmin=-1, vmax=1,
                square=True, linewidths=0, cbar_kws={"shrink": 0.6, "label": "corr"},
                xticklabels=disp, yticklabels=disp, annot=annot,
                fmt=".2f", annot_kws={"size": 6}, ax=ax)
    for b in _group_boundaries(ordered):
        ax.axhline(b, color="#0d1117", lw=2)
        ax.axvline(b, color="#0d1117", lw=2)
    ax.set_title(title, pad=12)
    ax.tick_params(labelsize=7)
    plt.setp(ax.get_xticklabels(), rotation=90)
    _style_ticks(ax)
    return ax

full_corr = residuals.corr()
plot_corr_heatmap(full_corr,
    f"Residual correlations - full sample ({residuals.index.min().date()} "
    f"to {residuals.index.max().date()}, {WINDOW}d beta-adjusted)")
plt.tight_layout(); plt.show()

### Parameterized sub-matrix

Same heatmap, restricted to a chosen set of groups so a specific relationship isn't
drowned out by the full 45×45 grid. Below: **style factors vs US sectors** — the
classic "is value just an energy/financials bet?" question, now asked on
market-neutral residuals.

In [ ]:
def plot_group_heatmap(groups, annot=False):
    '''Heatmap restricted to tickers in `groups` (list of group names).'''
    tickers = [t for t in residuals.columns if GROUPS[t] in groups]
    corr = residuals[tickers].corr()
    return plot_corr_heatmap(corr, " + ".join(groups) + f"  ({WINDOW}d residuals)",
                             annot=annot)

plot_group_heatmap(["Style Factors", "US Sectors"], annot=True)
plt.tight_layout(); plt.show()

---
## 2 · Rolling pair time series

A single full-sample number hides the story. Two residual streams can be tightly
linked in a crisis and unrelated in calm — the relationship is dynamic. This plots
the **rolling correlation** of two residual streams through time, with a zero line and
light shading where the correlation crosses ±0.5 (rough "they're moving as one
regime" markers). Watch for sign flips: those are the moments a hedge stopped
hedging.

In [ ]:
def plot_rolling_pair(ticker_a, ticker_b, window=WINDOW, ax=None):
    '''Rolling correlation between two residual streams, with regime shading.'''
    if ax is None:
        _, ax = plt.subplots(figsize=(12, 4))
    rc = residuals[ticker_a].rolling(window).corr(residuals[ticker_b])
    ax.plot(rc.index, rc.values, color=ACCENT_CYAN, lw=1.4)
    ax.axhline(0, color="#8b949e", lw=0.8, ls="--")
    ax.fill_between(rc.index, rc.values, 0.5, where=(rc > 0.5),
                    color=ACCENT_CYAN,  alpha=0.18)
    ax.fill_between(rc.index, rc.values, -0.5, where=(rc < -0.5),
                    color=ACCENT_AMBER, alpha=0.18)
    ax.set_ylim(-1, 1)
    la, lb = LABELS.get(ticker_a, ticker_a), LABELS.get(ticker_b, ticker_b)
    ax.set_title(f"{ticker_a} ({la})  vs  {ticker_b} ({lb})  -  rolling {window}d residual corr")
    ax.set_ylabel("correlation")
    _style_ticks(ax)
    return ax

pairs = [("IWD", "IWF"), ("IWD", "MTUM"), ("XLK", "XLF"), ("GLD", "IWD")]
fig, axes = plt.subplots(len(pairs), 1, figsize=(12, 3.2 * len(pairs)))
for (a, b), ax in zip(pairs, axes):
    plot_rolling_pair(a, b, ax=ax)
plt.tight_layout(); plt.show()

---
## 3 · Average pairwise correlation — the dispersion story

**This is the headline chart.** For each date we take a trailing window of residuals
and compute the *average off-diagonal correlation* among the members of a group. A
high value means everything in the group is moving together (low dispersion, hard
stock/sector picking); a falling value means the group is decoupling into independent
bets (high dispersion — a stock-picker's market).

Plotting all groups on one axis makes the divergence legible: the question for 2026 is
whether intra-sector and intra-factor correlations are collapsing while others hold.
The dashed lines mark each series' own long-run mean, so you can see who is currently
rich or cheap to its own history. We also add the **cross-group** factors-vs-sectors
average as a structural reference.

In [ ]:
def avg_pairwise(cols, window=WINDOW):
    '''Time series of mean off-diagonal rolling correlation among `cols`.'''
    cols = [c for c in cols if c in residuals.columns]
    sub  = residuals[cols]
    n    = len(cols)
    out  = pd.Series(index=sub.index, dtype=float)
    # Rolling correlation matrix per window endpoint; average the upper triangle.
    roll = sub.rolling(window)
    iu = np.triu_indices(n, k=1)
    for end in range(window - 1, len(sub)):
        w = sub.iloc[end - window + 1: end + 1]
        c = np.corrcoef(w.values, rowvar=False)
        out.iloc[end] = np.nanmean(c[iu])
    return out

def avg_pairwise_cross(cols_a, cols_b, window=WINDOW):
    '''Mean correlation of every a-in-A against every b-in-B (cross-block).'''
    a = [c for c in cols_a if c in residuals.columns]
    b = [c for c in cols_b if c in residuals.columns]
    out = pd.Series(index=residuals.index, dtype=float)
    for end in range(window - 1, len(residuals)):
        w = residuals.iloc[end - window + 1: end + 1]
        c = np.corrcoef(w[a + b].values, rowvar=False)
        block = c[:len(a), len(a):]
        out.iloc[end] = np.nanmean(block)
    return out

sectors = [t for t in residuals.columns if GROUPS[t] == "US Sectors"]
factors = [t for t in residuals.columns if GROUPS[t] == "Style Factors"]
intl    = [t for t in residuals.columns if GROUPS[t] == "International"]
cmdty   = [t for t in residuals.columns if GROUPS[t] == "Commodities"]

series = {
    "S&P sectors":        (avg_pairwise(sectors),  ACCENT_CYAN),
    "Style factors":      (avg_pairwise(factors),  ACCENT_AMBER),
    "International":       (avg_pairwise(intl),     "#9d7bff"),
    "Commodities":        (avg_pairwise(cmdty),    "#ff5d8f"),
    "Factors x Sectors":  (avg_pairwise_cross(factors, sectors), "#5fd35f"),
}

fig, ax = plt.subplots(figsize=(13, 6))
for name, (s, color) in series.items():
    ax.plot(s.index, s.values, label=name, color=color, lw=1.5)
    ax.axhline(s.mean(), color=color, lw=0.8, ls=":", alpha=0.7)
ax.axhline(0, color="#8b949e", lw=0.8)
ax.set_title(f"Average within-group residual correlation ({WINDOW}d trailing)  -  "
             "dotted = each series' long-run mean")
ax.set_ylabel("mean pairwise correlation")
ax.legend(loc="upper right", framealpha=0.3, ncol=2)
_style_ticks(ax)
plt.tight_layout(); plt.show()

---
## 4 · Regime comparison

Structure is not static, so compare it across explicitly chosen windows. For each
date range we recompute the residual correlation matrix and lay the heatmaps side by
side, then show a **difference matrix** (regime 2 − regime 1). The difference map is
where the insight is: cyan cells are pairs that became *more* correlated, amber cells
pairs that decoupled. The example contrasts the full 2024 against 2026 year-to-date.

In [ ]:
def compare_regimes(date_ranges, labels):
    '''Side-by-side residual correlation heatmaps for each range + a difference map.

    `date_ranges`: list of (start, end) string tuples. Difference shown is
    last regime minus first regime (requires exactly 2 ranges for the diff panel).'''
    mats = []
    for (s, e) in date_ranges:
        sub = residuals.loc[s:e]
        mats.append(sub.corr())

    n = len(date_ranges)
    ordered = _ordered(residuals.columns)
    disp = [LABELS[t] for t in ordered]
    bounds = _group_boundaries(ordered)

    ncols = n + (1 if n == 2 else 0)
    fig, axes = plt.subplots(1, ncols, figsize=(7.5 * ncols, 7.5))
    if ncols == 1:
        axes = [axes]

    for i, (m, lab) in enumerate(zip(mats, labels)):
        m = m.loc[ordered, ordered]
        sns.heatmap(m, cmap=DIVERGING, center=0, vmin=-1, vmax=1, square=True,
                    xticklabels=disp, yticklabels=disp,
                    cbar_kws={"shrink": 0.5}, ax=axes[i])
        axes[i].set_title(lab, pad=10)
        for b in bounds:
            axes[i].axhline(b, color="#0d1117", lw=1.5)
            axes[i].axvline(b, color="#0d1117", lw=1.5)
        axes[i].tick_params(labelsize=6)
        plt.setp(axes[i].get_xticklabels(), rotation=90)
        _style_ticks(axes[i])

    if n == 2:
        diff = (mats[1].loc[ordered, ordered] - mats[0].loc[ordered, ordered])
        ax = axes[-1]
        sns.heatmap(diff, cmap=DIVERGING, center=0, vmin=-1, vmax=1, square=True,
                    xticklabels=disp, yticklabels=disp,
                    cbar_kws={"shrink": 0.5, "label": "Δ corr"}, ax=ax)
        ax.set_title(f"Difference: {labels[1]} - {labels[0]}", pad=10)
        for b in bounds:
            ax.axhline(b, color="#0d1117", lw=1.5)
            ax.axvline(b, color="#0d1117", lw=1.5)
        ax.tick_params(labelsize=6)
        plt.setp(ax.get_xticklabels(), rotation=90)
        _style_ticks(ax)
    plt.tight_layout(); plt.show()
    return mats

_ = compare_regimes(
    [("2024-01-01", "2024-12-31"), ("2026-01-01", end_resolved)],
    ["2024 full year", "2026 YTD"],
)

---
*Built for personal research. Residual streams are beta-adjusted excess returns
(α + ε) from a rolling univariate OLS vs SPY — correlations here are factor-level,
not market co-movement. Tune `WINDOW`, `START_DATE`, and the `UNIVERSE` dict in CONFIG
and re-run; set `FORCE_REFRESH = True` to pull fresh prices.*